# Projet 3 : Support au Diagnostic Médical (Cleveland Heart Disease)
**Cours d'IA Avancée — Dr. Azer Hasnaoui (ING GLSI / AI-DS)**

Ce notebook présente la démarche complète étape par étape selon les séances du cours :
- **Séance 7** : Prétraitement (Imputation KNN, One-Hot drop='first', StandardScaler) & ACP
- **Séance 2** : Méthodes d'ensemble (Random Forest & XGBoost)
- **Séance 6** : Réseau de neurones profond (PyTorch DeepMLP)
- **Séance 8** : Validation croisée stratifiée (K=5), métriques médicales, Matrice de confusion, Courbe ROC et Boxplot

## 1. Chargement et Exploration des Données (Séance 7)

In [ ]:
from preprocessing import load_data

# Chargement du jeu de données
X, y = load_data()
print(f"Dimensions : {X.shape[0]} patients, {X.shape[1]} variables")
print(f"Classes : {(y == 0).sum()} Sains (0), {(y == 1).sum()} Malades (1)")

## 2. Prétraitement et Réduction de Dimension ACP (Séance 7)

In [ ]:
from preprocessing import get_preprocessor, run_pca

# Prétraitement global pour l'ACP
prep = get_preprocessor()
X_proc = prep.fit_transform(X)
raw_names = prep.get_feature_names_out()
feature_names = [f.split('__')[-1] for f in raw_names]

# Critère d'Arpenteur (Scree Plot >= 90%)
n_comp, var = run_pca(X_proc, threshold=0.90)
print(f"ACP : {n_comp} composantes capturent {var*100:.1f}% de la variance.")

## 3. Validation Croisée Stratifiée & Évaluation des Modèles (Séance 8)

In [ ]:
import numpy as np
from evaluate import evaluate_models, plot_evaluation

# Évaluation rigoureuse sans fuite de données
scores, y_true, y_probs, y_preds = evaluate_models(X, y)

print('--- RÉSULTATS DE LA VALIDATION CROISÉE (5 PLIS) ---')
for model_name, metrics in scores.items():
    rec = f"{np.mean(metrics['recall']):.3f} +/- {np.std(metrics['recall']):.3f}"
    prec = f"{np.mean(metrics['precision']):.3f} +/- {np.std(metrics['precision']):.3f}"
    f1 = f"{np.mean(metrics['f1']):.3f} +/- {np.std(metrics['f1']):.3f}"
    auc = f"{np.mean(metrics['roc_auc']):.3f} +/- {np.std(metrics['roc_auc']):.3f}"
    print(f"{model_name:<15} | Rappel: {rec} | Précision: {prec} | F1: {f1} | AUC: {auc}")

plot_evaluation(scores, y_true, y_probs, y_preds)

## 4. Importance des Facteurs de Risque Cliniques (Séance 2)

In [ ]:
from models import get_xgboost
from evaluate import plot_feature_importance

xgb_model = get_xgboost()
xgb_model.fit(X_proc, y)
plot_feature_importance(xgb_model, feature_names)